In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("YELP_API_KEY")

print (f"This is the key: {api_key}")   


This is the key: _TZZJbs55ENUDK2lRDU0lwmVf1DodAq4ja16ahgD9C_dd1WA8o1bEiCBetWi9cmFS9tdWiH5GKzAUKSh97-RjA62cALkEmAT4VslMrSdO1CzE8FFYVQdMZxw2-JLaHYx


In [5]:
# ✅ Yelp API Pull (Jupyter Version)

from pathlib import Path
from datetime import datetime
import os, json, requests
from dotenv import load_dotenv

# For Jupyter, use cwd walk-up instead
while not Path("README.md").exists() and Path.cwd() != Path("/"):
    os.chdir("..")  # Keep going up until we hit project root (assumes README is there)

# Now safely set paths
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Make sure you’ve exported your key in terminal before launching Jupyter:
# export YELP_API_KEY='your-key'
load_dotenv()
YELP_API_KEY = os.getenv("YELP_API_KEY")

headers = {
    "Authorization": f"Bearer {YELP_API_KEY}"
}

params = {
    "term": "asian food",
    "location": "Irvine, CA",
    "limit": 20,
    "sort_by": "rating"
}

response = requests.get("https://api.yelp.com/v3/businesses/search", headers=headers, params=params)

if response.status_code == 200:
    businesses = response.json().get("businesses", [])
    


    # Save as newline-delimited JSON
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # filename = f"yelp_sample_{timestamp}.json" old line where it would save it in the current directory
    filename = raw_dir / f"yelp_sample_{timestamp}.json"


    with open(filename, "w") as f:
        for business in businesses:
            json.dump(business, f)
            f.write("\n")
    
    print(f"✅ JSON file saved as {filename}")
else:
    print("❌ Yelp API call failed:", response.status_code, response.text)


✅ JSON file saved as data/raw/yelp_sample_20250619_202744.json


In [ ]:
from pathlib import Path
import os
import glob
from google.cloud import storage

# Dynamically move to project root (assuming README.md is there as a marker)
while not Path("README.md").exists() and Path.cwd() != Path("/"):
    os.chdir("..")

# Now find latest .json in project-root-relative data/raw/
json_files = glob.glob("data/raw/*.json")

if not json_files:
    raise FileNotFoundError("No JSON files found in data/raw/. Make sure you've run the API pull.")

latest_file = max(json_files, key=os.path.getctime)

# Upload to GCS
bucket_name = "daily-restaurant-insights-bucket"  # ← change this
destination_blob_path = f"raw/{os.path.basename(latest_file)}"

client = storage.Client()
bucket = client.bucket(bucket_name)
blob = bucket.blob(destination_blob_path)
blob.upload_from_filename(latest_file)

print(f"✅ Uploaded {latest_file} to gs://{bucket_name}/{destination_blob_path}")


ValueError: max() arg is an empty sequence